In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *
import random

# 1. Initialize Spark Session
spark = SparkSession.builder \
    .appName("Supercell_MonthEnd_Closing") \
    .getOrCreate()

# 2. จำลองข้อมูล (Mock Data Generation)
def generate_mock_data(num_records=10000):
    currencies = ['USD', 'JPY', 'EUR', 'AUD', 'GBP', 'THB']
    items = ['Gems Pack', 'Gold Pass', 'Skin Bundle', 'Resource Crate']
    
    data = []
    for i in range(num_records):
        tx_id = f"TXN-{i+10000}"
        sc_id = f"SCID-{random.randint(100, 999)}"
        item = random.choice(items)
        currency = random.choice(currencies)
        # สุ่มราคาตามความเหมาะสมของสกุลเงิน
        amount = round(random.uniform(0.99, 99.99), 2) if currency != 'JPY' else random.randint(100, 10000)
        
        data.append((tx_id, sc_id, item, amount, currency, "2026-02-25"))
    return data

# สร้าง DataFrame สำหรับ Transactions
schema = StructType([
    StructField("transaction_id", StringType(), False),
    StructField("supercell_id", StringType(), False),
    StructField("item_name", StringType(), False),
    StructField("amount", DoubleType(), False),
    StructField("currency", StringType(), False),
    StructField("event_date", StringType(), False)
])

df_tx = spark.createDataFrame(generate_mock_data(), schema)

# 3. สร้างตารางอัตราแลกเปลี่ยน (Exchange Rates to USD)
rates_data = [
    ('USD', 1.0),
    ('JPY', 0.0067),
    ('EUR', 1.08),
    ('AUD', 0.65),
    ('GBP', 1.27),
    ('THB', 0.028)
]
df_rates = spark.createDataFrame(rates_data, ["currency", "rate_to_usd"])

# 4. ขั้นตอนการปิดงวดบัญชี (Month-end Processing)
# - รวมข้อมูลการซื้อเข้ากับอัตราแลกเปลี่ยน
# - คำนวณรายได้สุทธิเป็น USD (Base Currency ของบริษัท)
closing_report = df_tx.join(df_rates, on="currency", how="left") \
    .withColumn("amount_usd", F.round(F.col("amount") * F.col("rate_to_usd"), 2))

# 5. สรุปผลรายได้แยกตามประเภทไอเทม
summary_report = closing_report.groupBy("item_name") \
    .agg(
        F.count("transaction_id").alias("total_transactions"),
        F.sum("amount_usd").alias("total_revenue_usd")
    ).orderBy(F.desc("total_revenue_usd"))

# แสดงผล
print("--- Month-End Revenue Summary (USD) ---")
summary_report.show()

# 6. ตรวจสอบ User Activity (การใช้ Items)
# สมมติฐาน: ในระบบจริงจะมีการส่ง Event 'Item_Used' มาด้วย
print("--- Sample Transaction Logs ---")
closing_report.select("supercell_id", "item_name", "amount", "currency", "amount_usd").show(5)